In [40]:
%pip install -q -U \
    "trl[peft,quantization]" \
    transformers \
    accelerate \
    datasets \
    huggingface-hub \
    pyarrow \
    sentencepiece

In [41]:
%pip install -q "fsspec[http]==2025.12.0"

In [42]:
import os
import transformers
import torch
from google.colab import userdata
from datasets import load_dataset
from trl import SFTTrainer
from peft import LoraConfig
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    GemmaTokenizer
)

In [43]:
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [44]:
model_id = "google/gemma-2b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)


In [45]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token=os.environ["HF_TOKEN"])
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"":0},
    token=os.environ["HF_TOKEN"])


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

In [46]:
text = "Qutoe: Imagination is more"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Qutoe: Imagination is more important than knowledge. - Albert Einstein

Answer:

Step 1/2
First, we need to understand what is meant by "imagination" and "knowledge." Imagination refers to the ability to create and visualize ideas, while knowledge refers to the accumulation of information and facts.

Step 2/2
Next


In [47]:
text = "Qutoe: Imagination is more"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Qutoe: Imagination is more important than knowledge. - Albert Einstein

Answer:

Step 1/2
First, we


In [48]:
os.environ["WANDB_DISABLED"] = "false"

In [49]:
lora_config = LoraConfig(
    r = 8,
    target_modules = ["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type = "CAUSAL_LM",
    lora_alpha = 32,
)

In [50]:
from datasets import load_dataset

data = load_dataset("Abirate/english_quotes")
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

In [51]:
data['train']['quote']

Column(['“Be yourself; everyone else is already taken.”', "“I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me at my worst, then you sure as hell don't deserve me at my best.”", "“Two things are infinite: the universe and human stupidity; and I'm not sure about the universe.”", '“So many books, so little time.”', '“A room without books is like a body without a soul.”', ...])

In [52]:
def formatting_func(example):
  return {
        "text": (
            f"Quote: {example['quote']}\n"
            f"Author: {example['author']}"
            f"{tokenizer.eos_token}"
        )
  }

In [53]:
trainer = SFTTrainer(
    model=model,
    train_dataset=data["train"],
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=2,
        max_steps=100,
        learning_rate=2e-4,
        fp16=True,
        bf16=False,
        logging_steps=1,
        output_dir="outputs",
        optim="paged_adamw_8bit"
    ),
    formatting_func=formatting_func,
    peft_config=lora_config,
)

In [54]:
from collections import Counter

# Check trainable parameter types
print("Before:", Counter(
    str(p.dtype)
    for p in trainer.model.parameters()
    if p.requires_grad
))

# Convert trainable LoRA parameters to FP32
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

print("After:", Counter(
    str(p.dtype)
    for p in trainer.model.parameters()
    if p.requires_grad
))

trainer.train()

Before: Counter({'torch.bfloat16': 252})
After: Counter({'torch.float32': 252})


Step,Training Loss
1,2.679437
2,1.702476
3,2.587360
4,2.810028
5,2.448046
6,2.474279
7,2.907205
8,2.160104
9,3.168459
10,2.162688


TrainOutput(global_step=100, training_loss=2.103392389416695, metrics={'train_runtime': 131.3179, 'train_samples_per_second': 3.046, 'train_steps_per_second': 0.762, 'total_flos': 189744345784320.0, 'train_loss': 2.103392389416695, 'epoch': 0.1594896331738437})

In [56]:
text = "Qutoe: A woman is like a tea bag,"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = model.generate(**inputs, max_new_tokens=30)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Qutoe: A woman is like a tea bag, you can't tell how strong she is until you put her in hot water.

Quote: I'm not a feminist. I'
